## Task 1

### Step 1

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 500000
cities = ["Berlin", "Tokyo", "New York", "Paris", "Warsaw", "London", "Istanbul", "Rome", "Seoul", "Sydney"]

data = {
    "user_id": np.arange(1, n + 1),
    "city": np.random.choice(cities, n),
    "score": np.random.uniform(0, 100, n),
    "active": np.random.choice([True, False], n),
    "signup_date": pd.to_datetime('today') - pd.to_timedelta(np.random.randint(0, 3*365, n), unit='D'),
    "age": np.random.randint(18, 81, n),
    "sessions": np.random.randint(0, 501, n),
    "revenue": np.random.uniform(0, 1000, n)
}

df = pd.DataFrame(data)
df.head()

,user_id,city,score,active,signup_date,age,sessions,revenue
0,1,Istanbul,43.689490,False,2026-02-02 02:16:24.448184,71,323,179.164861
1,2,Paris,97.403462,True,2025-06-03 02:16:24.448184,39,352,614.893856
2,3,Rome,66.148238,False,2025-11-09 02:16:24.448184,23,279,217.069799
3,4,Warsaw,21.993544,False,2025-05-23 02:16:24.448184,20,129,195.459273
4,5,Istanbul,91.719953,True,2023-04-03 02:16:24.448184,67,199,617.349590


### Step 2

In [2]:
import pyarrow as pa
import pyarrow.parquet as pq

df.to_parquet("df.parquet", index = False)

parquet_file = pq.ParquetFile("df.parquet")

print(f"Number of row groups: {parquet_file.metadata.num_row_groups}")

print(f"Number of columns: {parquet_file.metadata.num_columns}, Number of rows: {parquet_file.metadata.num_rows}")

print(f"\nSchema (column names and types):\n{parquet_file.schema_arrow}")

row_group = parquet_file.metadata.row_group(0)
for i in range(row_group.num_columns):
    col = row_group.column(i)
    stats = col.statistics
    print(f"Column: {col.path_in_schema:10s} | "
          f"Type: {col.physical_type:8s} | "
          f"Compressed size: {col.total_compressed_size:>8d} bytes | "
          f"Min: {stats.min} | "
          f"Max: {stats.max} | "
          f"Null count: {stats.null_count}")

Number of row groups: 1
Number of columns: 8, Number of rows: 500000

Schema (column names and types):
user_id: int64
city: string
score: double
active: bool
signup_date: timestamp[ns]
age: int32
sessions: int32
revenue: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 974
Column: user_id    | Type: INT64    | Compressed size:  2280081 bytes | Min: 1 | Max: 500000 | Null count: 0
Column: city       | Type: BYTE_ARRAY | Compressed size:   251173 bytes | Min: Berlin | Max: Warsaw | Null count: 0
Column: score      | Type: DOUBLE   | Compressed size:  4279334 bytes | Min: 5.188445665327279e-05 | Max: 99.99983148609545 | Null count: 0
Column: active     | Type: BOOLEAN  | Compressed size:    62553 bytes | Min: False | Max: True | Null count: 0
Column: signup_date | Type: INT64    | Compressed size:   697142 bytes | Min: 2023-03-08 02:16:24.448184 | Max: 2026-03-06 02:16:24.448184 | Null count: 0
Column: age        | Type: INT32    | 

### Step 3

In [3]:
import os
df.to_csv("df.csv", index = False)
csv_size = os.path.getsize("df.csv")/1024
parquet_size = os.path.getsize("df.parquet")/1024
compression_ratio = csv_size/parquet_size

print(f"CSV size: {csv_size:.2f} KB")
print(f"Parquet size: {parquet_size:.2f} KB")
print(f"Compression ratio: {compression_ratio:.2f}")

CSV size: 44094.39 KB
Parquet size: 12495.94 KB
Compression ratio: 3.53


### Step 4

#### Parquet metadata provides physical type, compressed size, any available statistics, predicate pushdown, and location for each column(chunks) in row groups. These statistics enable predicate pushdown at the file level. It matters because these are advantages where we have to scan the entire file to know basic facts about data

## Task 2

### Step 1

In [4]:
%time parquet_file = pd.read_parquet("df.parquet")

CPU times: total: 141 ms
Wall time: 115 ms


### Step 2

In [5]:
%time subset = pd.read_parquet("df.parquet", columns=["city", "revenue"])

CPU times: total: 93.8 ms
Wall time: 64.3 ms


### Step 3

In [6]:
%%timeit
csv_file = pd.read_csv("df.csv")
subset = csv_file[["city", "revenue"]]

694 ms ± 24.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


### Step 4

#### Beacuse each column chunk in parquet is stored independently. That is why this query that only needs 2 columns out of 8 can skip the other 6 entirely. 

## Task 3

### Step 1

In [7]:
import pyarrow as pa
import pyarrow.parquet as pq

%time arrow_table = pq.read_table("df.parquet", filters=[("age", ">", 50)])

CPU times: total: 78.1 ms
Wall time: 51.7 ms


### Step 2

In [8]:
%%time 
df = pd.read_parquet("df.parquet")
filtered_df = df[df["age"]>50]

CPU times: total: 203 ms
Wall time: 92 ms


### Step 3

In [9]:
print(f"PyArrow returned row number:{len(arrow_table)}")
print(f"Pandas returned row number:{len(filtered_df)}")

PyArrow returned row number:237812
Pandas returned row number:237812


#### In both approaches, number of returned rows are same. But time in parquet is less than time in pandas 
#### Because when reading a Parquet file with a filter, a smart engine uses the column statistics in the file's metadata to skip entire row groups. With pushdown, row groups whose age column has a maximum value below 50 are never read at all. Without pushdown, the engine  reads the entire file into memory and then apply the filter in pandas

### Step 4

In [10]:
import duckdb
%time result = duckdb.sql("SELECT * FROM 'df.parquet' WHERE age > 50").df()

CPU times: total: 141 ms
Wall time: 93.3 ms


In [11]:
print(f"DuckDB reurned row number:{len(result)}")

DuckDB reurned row number:237812


#### In each of them, the number of returned rows are same. But :                             time in pandas > time in pyarrow > time in duckdb

## Task 4

### Query 1

In [12]:
import time

start_p = time.time()
# Pandas
records_per_city_p = df["city"].value_counts()
time_p = time.time() - start_p

print(f"Pandas time: {time_p:.4f} s")
print(records_per_city_p.head())

Pandas time: 0.0331 s
city
Warsaw      50216
New York    50113
Sydney      50061
London      50059
Berlin      50059
Name: count, dtype: int64


In [13]:
start_d = time.time()
# DuckDB:
records_per_city_d = duckdb.sql("SELECT city, COUNT(*) as count FROM df GROUP BY city").df()
time_d = time.time() - start_d

print(f"DuckDB time: {time_d:.4f} s")
print(records_per_city_d.head())

DuckDB time: 0.0152 s
       city  count
0    Warsaw  50216
1     Tokyo  49970
2     Paris  49931
3    London  50059
4  New York  50113


### Query 2

In [14]:
start_p = time.time()
# Pandas
avg_score_by_city_p = df.groupby("city")["score"].mean().sort_values(ascending=False)
time_p = time.time() - start_p

print(f"Pandas time: {time_p:.4f} s")
print(avg_score_by_city_p.head())

Pandas time: 0.0464 s
city
London      50.311448
Warsaw      50.151899
Tokyo       50.135854
Istanbul    50.075511
Rome        50.063780
Name: score, dtype: float64


In [15]:
start_d = time.time()
# DuckDB:
avg_score_by_city_d = duckdb.sql("SELECT city, AVG(score) as Average FROM df GROUP BY city ORDER BY AVG(score) DESC").df()
time_d = time.time() - start_d

print(f"DuckDB time: {time_d:.4f} s")
print(avg_score_by_city_d.head())

DuckDB time: 0.0159 s
       city    Average
0    London  50.311448
1    Warsaw  50.151899
2     Tokyo  50.135854
3  Istanbul  50.075511
4      Rome  50.063780


### Query 3

In [16]:
start_p = time.time()
# Pandas
active_users = df[df["active"] == True]

result_p = active_users.groupby("city")["score"].apply(lambda x: (x > 75).mean() * 100)
time_p = time.time() - start_p

print(f"Pandas time: {time_p:.4f} s")
print(result_p.head())

Pandas time: 0.0620 s
city
Berlin      24.981993
Istanbul    25.064785
London      25.538499
New York    24.764614
Paris       25.431900
Name: score, dtype: float64


In [17]:
start_d = time.time()
# DuckDB:
active_users = duckdb.sql("""SELECT city, AVG(score)*100 as Percentage FROM df 
                                   WHERE active = True AND score > 75
                                   GROUP BY city 
                                   ORDER BY Percentage DESC""").df()

time_d = time.time() - start_d

print(f"DuckDB time: {time_d:.4f} s")
print(active_users.head())

DuckDB time: 0.0205 s
       city   Percentage
0    Warsaw  8765.123284
1  Istanbul  8759.367961
2  New York  8756.374000
3    Berlin  8756.260760
4    Sydney  8756.074424


### Query 4

In [18]:
start_p = time.time()
# Pandas
top_10_p = df.sort_values(['city', 'score'], ascending=[True, False]) \
                  .groupby('city') \
                  .head(10)
time_p = time.time() - start_p

print(f"Pandas vaxtı: {time_p:.4f} s")
print(top_10_p.head(12))

Pandas vaxtı: 0.4383 s
        user_id      city      score  active                signup_date  age  \
222164   222165    Berlin  99.994823    True 2025-05-01 02:16:24.448184   75   
474291   474292    Berlin  99.992728    True 2025-01-03 02:16:24.448184   48   
96404     96405    Berlin  99.992615   False 2024-07-14 02:16:24.448184   57   
348899   348900    Berlin  99.992278   False 2024-12-18 02:16:24.448184   67   
198655   198656    Berlin  99.988107   False 2025-11-25 02:16:24.448184   47   
402547   402548    Berlin  99.985122    True 2023-06-25 02:16:24.448184   27   
431887   431888    Berlin  99.983603    True 2025-01-25 02:16:24.448184   28   
433778   433779    Berlin  99.978958   False 2023-07-09 02:16:24.448184   79   
326997   326998    Berlin  99.978825   False 2025-07-21 02:16:24.448184   42   
8135       8136    Berlin  99.977353   False 2024-06-04 02:16:24.448184   41   
239739   239740  Istanbul  99.998692    True 2025-09-21 02:16:24.448184   74   
112236   112237  

In [19]:
start_d = time.time()
# DuckDB:
top_10_d = duckdb.sql("""SELECT city, score FROM (
    SELECT city, score, 
           ROW_NUMBER() OVER (PARTITION BY city ORDER BY score DESC) as rank
    FROM df) WHERE rank <= 10""").df()

time_d = time.time() - start_d

print(f"DuckDB time: {time_d:.4f} s")
print(top_10_d.head())

DuckDB time: 0.0949 s
    city      score
0  Tokyo  99.999831
1  Tokyo  99.996193
2  Tokyo  99.993404
3  Tokyo  99.993183
4  Tokyo  99.993058


### Query 5

In [20]:
start_p = time.time()

df_sorted = df.sort_values(['city', 'user_id'])

df_sorted['running_total'] = df_sorted.groupby('city')['score'].cumsum()
time_p = time.time() - start_p

print(f"Pandas vaxtı: {time_p:.4f} s")
print(df_sorted[['city', 'user_id', 'score', 'running_total']].head(10))

Pandas vaxtı: 0.1857 s
       city  user_id      score  running_total
21   Berlin       22  68.129047      68.129047
25   Berlin       26  69.119490     137.248537
51   Berlin       52  89.111342     226.359879
78   Berlin       79  32.234163     258.594042
83   Berlin       84  67.520144     326.114186
87   Berlin       88  28.858131     354.972316
91   Berlin       92  25.708829     380.681145
101  Berlin      102  54.317300     434.998445
110  Berlin      111  21.177261     456.175706
114  Berlin      115  19.584962     475.760668


In [21]:
start_d = time.time()
# DuckDB:
running_total = duckdb.sql("""SELECT city, user_id, score,SUM(score) 
                           OVER (PARTITION BY city ORDER BY user_id) as running_total FROM df
                           ORDER BY city, user_id""").df()

time_d = time.time() - start_d

print(f"DuckDB time: {time_d:.4f} s")
print(running_total.head())

DuckDB time: 0.2580 s
     city  user_id      score  running_total
0  Berlin       22  68.129047      68.129047
1  Berlin       26  69.119490     137.248537
2  Berlin       52  89.111342     226.359879
3  Berlin       79  32.234163     258.594042
4  Berlin       84  67.520144     326.114186


#### Which tool was easier to express each query in?
#### DuckDb
#### Which was faster?
#### Mostly DuckDB
#### For which queries did the difference matter most?
#### In Query 3 and Query 4

## Task 5

### Step 1

In [22]:
import pandas as pd
import numpy as np

n_students = 10000

faculties = ['Computer Science', 'Mechanical Engineering', 'Medicine', 'Law', 'Architecture', 'Business Administration']
cities = ['London', 'San Francisco', 'Berlin', 'Tokyo', 'Toronto', 'Sydney']

data = {
    'student_id': [f"STD-{i:05d}" for i in range(n_students)],
    'name': [f"Student_{i}" for i in range(n_students)],
    'faculty': np.random.choice(faculties, n_students),
    'city': np.random.choice(cities, n_students),
    'attendance_rate': np.random.uniform(50, 100, n_students).round(2),
    'midterm_score': np.random.randint(30, 100, n_students),
    'final_score': np.random.randint(30, 100, n_students),
    'study_hours_per_week': np.random.randint(5, 40, n_students),
    'is_scholarship': np.random.choice([True, False], n_students, p=[0.2, 0.8]) 
}


student_df = pd.DataFrame(data)


student_df.head()

,student_id,name,faculty,city,attendance_rate,midterm_score,final_score,study_hours_per_week,is_scholarship
0,STD-00000,Student_0,Computer Science,Berlin,51.56,97,71,34,False
1,STD-00001,Student_1,Mechanical Engineering,Berlin,87.93,38,60,33,False
2,STD-00002,Student_2,Medicine,Tokyo,64.52,40,99,9,False
3,STD-00003,Student_3,Medicine,London,92.40,38,48,27,False
4,STD-00004,Student_4,Medicine,San Francisco,72.94,43,57,18,False


In [23]:
import pyarrow as pa
import pyarrow.parquet as pq

arrow_from_student_df = pa.Table.from_pandas(student_df)

### Step 2

In [24]:
print(arrow_from_student_df.schema)

student_id: string
name: string
faculty: string
city: string
attendance_rate: double
midterm_score: int32
final_score: int32
study_hours_per_week: int32
is_scholarship: bool
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 1346


In [25]:
student_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   student_id            10000 non-null  object 
 1   name                  10000 non-null  object 
 2   faculty               10000 non-null  object 
 3   city                  10000 non-null  object 
 4   attendance_rate       10000 non-null  float64
 5   midterm_score         10000 non-null  int32  
 6   final_score           10000 non-null  int32  
 7   study_hours_per_week  10000 non-null  int32  
 8   is_scholarship        10000 non-null  bool   
dtypes: bool(1), float64(1), int32(3), object(4)
memory usage: 517.7+ KB


#### Columns that types are string in Arrow table are object in Pandas. And type of attendance_rate in Arrow table is double, in Pandas is folat 64

### Step 3

In [26]:
students_parquet = pq.write_table(arrow_from_student_df, 'students.parquet')
new_arrow_table = pq.read_table('students.parquet')

### Step 4

In [27]:
# Arrow Table -> Pandas DataFrame
df_recovered = new_arrow_table.to_pandas()

is_equal = student_df.equals(df_recovered)
print(f"Does the recovered data match the original? {is_equal}")


Does the recovered data match the original? True


### Step 5

In [28]:
df_traditional = pd.read_parquet('students.parquet')

df_arrow_backend = pd.read_parquet('students.parquet', dtype_backend="pyarrow")

print("--- Traditional Pandas Dtypes ---")
print(df_traditional.dtypes.head(4))

print("\n--- Arrow-backed Pandas Dtypes ---")
print(df_arrow_backend.dtypes.head(4))

--- Traditional Pandas Dtypes ---
student_id    object
name          object
faculty       object
city          object
dtype: object

--- Arrow-backed Pandas Dtypes ---
student_id    string[pyarrow]
name          string[pyarrow]
faculty       string[pyarrow]
city          string[pyarrow]
dtype: object


### Step 6

#### Arrow acts as a universal columnar memory format that eliminates the need for data serialization between tools. It allows Parquet (columnar on disk) to be mapped directly into RAM, where pandas (analysis) and DuckDB (SQL) can both process the exact same memory buffers simultaneously. This "zero-copy" connection ensures that data moves from storage to calculation at lightning speed without wasting CPU or RAM on reformatting.. 